# SignGuard LK - YOLOv8 Training Notebook

This notebook is the training and evaluation workspace for SignGuard LK, a Sri Lankan road-sign intelligence system for safer driving. It is organized to keep data loading, configuration, training, evaluation, inference simulation, and export steps separate.

Run it in Google Colab for GPU training, or locally if CUDA/PyTorch is configured.


## 1. Environment Setup

Installs the training dependencies and prepares reusable imports. The project app itself uses the smaller dependency set in `requirements.txt`; this notebook also uses `roboflow` for dataset download.

In [ ]:
!pip install -q ultralytics roboflow pyyaml pandas matplotlib opencv-python

import os
import glob
import shutil
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from ultralytics import YOLO

try:
    from google.colab import files, userdata
    from google.colab.patches import cv2_imshow
    IN_COLAB = True
except Exception:
    files = None
    userdata = None
    cv2_imshow = None
    IN_COLAB = False

print(f"Running in Colab: {IN_COLAB}")

## 2. Training Configuration

Adjust these values before training. The default model name matches the app export path and report artifact paths used later.

In [ ]:
PROJECT_ROOT = Path.cwd()
RUN_NAME = "sri_lanka_yolo_model"
BASE_MODEL = "yolov8m.pt"  # yolov8s.pt is faster; yolov8m.pt/yolov8l.pt usually improve accuracy
IMAGE_SIZE = 768            # larger sizes help small traffic signs, with slower training/inference
EPOCHS = 100
BATCH_SIZE = 16
DEVICE = 0  # use 0 for GPU in Colab, or "cpu" when no GPU is available
CONFIDENCE_THRESHOLD = 0.40
IOU_THRESHOLD = 0.45
PATIENCE = 25
SEED = 42

ROBOFLOW_WORKSPACE = "traffic-signs-in-sri-lanka"
ROBOFLOW_PROJECT = "traffic-signs-in-sri-lanka"
ROBOFLOW_VERSION = 4

RUN_DIR = PROJECT_ROOT / "runs" / "detect" / RUN_NAME
BEST_MODEL_PATH = RUN_DIR / "weights" / "best.pt"
APP_MODEL_PATH = PROJECT_ROOT / "best.pt"

print("Project root:", PROJECT_ROOT)
print("Training run:", RUN_NAME)
print("Base model:", BASE_MODEL)


## 3. Download Dataset

Do not hardcode API keys in the notebook. In Colab, add `ROBOFLOW_API_KEY` under Secrets. Locally, set it as an environment variable before running.

In [ ]:
from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY")
if not api_key and IN_COLAB and userdata:
    try:
        api_key = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        api_key = None

if not api_key:
    raise RuntimeError("Set ROBOFLOW_API_KEY in Colab Secrets or as an environment variable.")

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download("yolov8")
DATASET_DIR = Path(dataset.location)
YAML_PATH = DATASET_DIR / "data.yaml"

print("Dataset downloaded to:", DATASET_DIR)
print("Dataset config:", YAML_PATH)

## 4. Normalize Dataset YAML

Roboflow exports sometimes contain relative paths. This cell rewrites the YAML with absolute train/validation/test image paths so training is reliable from any working directory.

In [ ]:
with YAML_PATH.open("r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

data_yaml["train"] = str(DATASET_DIR / "train" / "images")
data_yaml["val"] = str(DATASET_DIR / "valid" / "images")
data_yaml["test"] = str(DATASET_DIR / "test" / "images")

with YAML_PATH.open("w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"Classes: {data_yaml['nc']}")
print("Train:", data_yaml["train"])
print("Valid:", data_yaml["val"])
print("Test:", data_yaml["test"])

## 5. Dataset Health Check

Quickly checks image/label counts before spending GPU time on training.

In [ ]:
def count_files(folder, patterns):
    folder = Path(folder)
    total = 0
    for pattern in patterns:
        total += len(list(folder.glob(pattern)))
    return total

rows = []
for split, image_key in [("train", "train"), ("valid", "val"), ("test", "test")]:
    image_dir = Path(data_yaml[image_key])
    label_dir = image_dir.parent / "labels"
    rows.append({
        "split": split,
        "images": count_files(image_dir, ["*.jpg", "*.jpeg", "*.png"]),
        "labels": count_files(label_dir, ["*.txt"]),
        "image_dir": str(image_dir),
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

if (summary_df["images"] == 0).any():
    raise RuntimeError("At least one dataset split has no images. Check the Roboflow export.")

## 6. Class Balance Analysis

Accuracy often drops on rare classes. This cell counts labels per class so you can identify classes that need more examples or targeted augmentation.

In [ ]:
from collections import Counter

class_names = data_yaml.get("names", [])
if isinstance(class_names, dict):
    class_names = [class_names[i] for i in sorted(class_names)]

label_counts = Counter()
for split_key in ["train", "val", "test"]:
    label_dir = Path(data_yaml[split_key]).parent / "labels"
    for label_file in label_dir.glob("*.txt"):
        for line in label_file.read_text(encoding="utf-8").splitlines():
            if line.strip():
                class_id = int(line.split()[0])
                label_counts[class_id] += 1

class_balance_df = pd.DataFrame([
    {"class_id": i, "class_name": class_names[i] if i < len(class_names) else str(i), "labels": label_counts.get(i, 0)}
    for i in range(len(class_names))
]).sort_values("labels")

display(class_balance_df)

plt.figure(figsize=(14, 5))
plt.bar(class_balance_df["class_name"], class_balance_df["labels"])
plt.xticks(rotation=90)
plt.title("Label Count by Class")
plt.ylabel("Bounding boxes")
plt.tight_layout()
plt.show()

rare_classes = class_balance_df[class_balance_df["labels"] < class_balance_df["labels"].median() * 0.5]
print("Classes that may need more images/augmentation:")
display(rare_classes)

## 6. Train YOLOv8

Training output is saved under `runs/detect/sri_lanka_yolo_model/`.

In [ ]:
model = YOLO(BASE_MODEL)

train_results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    name=RUN_NAME,
    save=True,
    patience=PATIENCE,
    seed=SEED,
    pretrained=True,
    optimizer="AdamW",
    cos_lr=True,
    close_mosaic=10,
    hsv_h=0.015,
    hsv_s=0.70,
    hsv_v=0.40,
    degrees=8.0,
    translate=0.10,
    scale=0.50,
    shear=2.0,
    perspective=0.0005,
    fliplr=0.50,
    mosaic=1.0,
    mixup=0.10,
    copy_paste=0.10,
)

print("Best model:", BEST_MODEL_PATH)


## 7. Validate Model

Runs validation and prints the global object-detection metrics used in the report.

In [ ]:
best_model = YOLO(str(BEST_MODEL_PATH))
val_results = best_model.val(data=str(YAML_PATH), split="val")

metrics = val_results.results_dict
print("Validation metrics")
print(f"Precision:  {metrics.get('metrics/precision(B)', 0):.4f}")
print(f"Recall:     {metrics.get('metrics/recall(B)', 0):.4f}")
print(f"mAP50:      {metrics.get('metrics/mAP50(B)', 0):.4f}")
print(f"mAP50-95:   {metrics.get('metrics/mAP50-95(B)', 0):.4f}")

## 8. Test Set Evaluation

Use the test split for the final unbiased result after tuning is complete.

In [ ]:
test_results = best_model.val(data=str(YAML_PATH), split="test")

test_metrics = test_results.results_dict
print("Test metrics")
print(f"Precision:  {test_metrics.get('metrics/precision(B)', 0):.4f}")
print(f"Recall:     {test_metrics.get('metrics/recall(B)', 0):.4f}")
print(f"mAP50:      {test_metrics.get('metrics/mAP50(B)', 0):.4f}")
print(f"mAP50-95:   {test_metrics.get('metrics/mAP50-95(B)', 0):.4f}")

## 9. View Training Artifacts

Displays common YOLO output charts when they exist.

In [ ]:
def show_image(path, title=None, figsize=(12, 8)):
    path = Path(path)
    if not path.exists():
        print(f"Missing: {path}")
        return
    image = cv2.imread(str(path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

show_image(RUN_DIR / "results.png", "Training Metrics", figsize=(14, 10))
show_image(RUN_DIR / "confusion_matrix.png", "Confusion Matrix", figsize=(12, 10))
show_image(RUN_DIR / "F1_curve.png", "F1 Curve", figsize=(10, 8))
show_image(RUN_DIR / "val_batch0_labels.jpg", "Validation Labels", figsize=(15, 10))
show_image(RUN_DIR / "val_batch0_pred.jpg", "Validation Predictions", figsize=(15, 10))

## 10. Threshold Tuning

A better confidence threshold can improve practical accuracy by reducing false positives without hiding too many correct detections. Test several thresholds on validation data and choose the best tradeoff for the app.

In [ ]:
threshold_rows = []
for conf in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60]:
    result = best_model.val(data=str(YAML_PATH), split="val", conf=conf, iou=IOU_THRESHOLD, verbose=False)
    metrics = result.results_dict
    threshold_rows.append({
        "confidence": conf,
        "precision": metrics.get("metrics/precision(B)", 0),
        "recall": metrics.get("metrics/recall(B)", 0),
        "mAP50": metrics.get("metrics/mAP50(B)", 0),
        "mAP50-95": metrics.get("metrics/mAP50-95(B)", 0),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df["f1_estimate"] = 2 * threshold_df["precision"] * threshold_df["recall"] / (
    threshold_df["precision"] + threshold_df["recall"] + 1e-9
)
display(threshold_df.sort_values("f1_estimate", ascending=False))

best_threshold = threshold_df.sort_values("f1_estimate", ascending=False).iloc[0]
print(f"Recommended app confidence threshold: {best_threshold['confidence']:.2f}")

## 11. Model Size Comparison

Optional but useful for accuracy work: train a small set of YOLO sizes for fewer epochs, then pick the best accuracy/speed tradeoff before the final long training run.

In [ ]:
MODEL_CANDIDATES = ["yolov8s.pt", "yolov8m.pt"]
COMPARE_EPOCHS = 20
comparison_rows = []

for candidate in MODEL_CANDIDATES:
    candidate_name = f"compare_{Path(candidate).stem}"
    candidate_model = YOLO(candidate)
    candidate_model.train(
        data=str(YAML_PATH),
        epochs=COMPARE_EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        name=candidate_name,
        patience=8,
        seed=SEED,
        optimizer="AdamW",
        cos_lr=True,
    )
    candidate_best = PROJECT_ROOT / "runs" / "detect" / candidate_name / "weights" / "best.pt"
    candidate_eval = YOLO(str(candidate_best)).val(data=str(YAML_PATH), split="val", verbose=False)
    metrics = candidate_eval.results_dict
    comparison_rows.append({
        "model": candidate,
        "mAP50": metrics.get("metrics/mAP50(B)", 0),
        "mAP50-95": metrics.get("metrics/mAP50-95(B)", 0),
        "precision": metrics.get("metrics/precision(B)", 0),
        "recall": metrics.get("metrics/recall(B)", 0),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("mAP50-95", ascending=False)
display(comparison_df)

## 12. Error Mining Samples

This finds low-confidence predictions from the validation set. Review these images to discover labeling mistakes, confusing sign classes, night/blur problems, or classes that need more data.

In [ ]:
VAL_IMAGE_DIR = Path(data_yaml["val"])
val_images = sorted(glob.glob(str(VAL_IMAGE_DIR / "*.*")))[:80]
error_rows = []

for image_path in val_images:
    predictions = best_model.predict(source=image_path, conf=0.10, iou=IOU_THRESHOLD, verbose=False)
    for prediction in predictions:
        for box in prediction.boxes:
            confidence = float(box.conf[0])
            if confidence < 0.40:
                class_id = int(box.cls[0])
                error_rows.append({
                    "image": image_path,
                    "class": best_model.names[class_id],
                    "confidence": confidence,
                })

error_df = pd.DataFrame(error_rows).sort_values("confidence") if error_rows else pd.DataFrame(columns=["image", "class", "confidence"])
display(error_df.head(30))
print("Use these samples to improve labels or add similar images to the dataset.")

## 13. ADAS Inference Simulation

Tests one image from the test set with the same kind of speed-aware alert logic used by the SignGuard LK app.

In [ ]:
try:
    from src.data.road_signs import ROAD_SIGNS
except Exception:
    ROAD_SIGNS = {}

CURRENT_SPEED = 75
SCHOOL_ZONE_ACTIVE = True
TEST_IMAGE_DIR = Path(data_yaml["test"])
image_list = sorted(glob.glob(str(TEST_IMAGE_DIR / "*.*")))

if not image_list:
    raise RuntimeError(f"No test images found in {TEST_IMAGE_DIR}")

def classify_adas_alert(class_name, speed, school_zone_active):
    label = ROAD_SIGNS.get(class_name, class_name)
    threat = "SIGN DETECTED"
    color = (45, 212, 191)

    if class_name.startswith("SLS-"):
        try:
            limit = int(class_name.split("-")[1])
            if speed > limit:
                return "Over Speeding!", "CRITICAL", (0, 0, 255)
        except Exception:
            pass

    if class_name == "DWS-33" and school_zone_active:
        if speed > 30:
            return "School Zone - Reduce Speed!", "CRITICAL", (0, 0, 255)
        return "School Zone - Speed OK", "SAFE", (0, 255, 0)

    if class_name == "PRS-01":
        if speed > 5:
            return "STOP Sign Ignored!", "CRITICAL", (0, 0, 255)
        return "Vehicle Stopped", "SAFE", (0, 255, 0)

    return label, threat, color

sample_image = image_list[0]
results = best_model.predict(source=sample_image, conf=CONFIDENCE_THRESHOLD)

for result in results:
    frame = result.orig_img.copy()
    for box in result.boxes:
        cls_id = int(box.cls[0])
        class_name = best_model.names[cls_id]
        display_name, threat_level, color = classify_adas_alert(class_name, CURRENT_SPEED, SCHOOL_ZONE_ACTIVE)
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        confidence = float(box.conf[0]) * 100
        label_text = f"{display_name} | {threat_level} | {confidence:.0f}%"
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label_text, (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    if IN_COLAB and cv2_imshow:
        cv2_imshow(frame)
    else:
        show_image(sample_image, "Sample source image")
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title("ADAS Inference Simulation")
        plt.show()

## 14. Export Model for the Desktop App

Copies the trained model to the repository root as `best.pt`, which is the path used by the application.

In [ ]:
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Best model not found: {BEST_MODEL_PATH}")

shutil.copy2(BEST_MODEL_PATH, APP_MODEL_PATH)
print("Exported app model to:", APP_MODEL_PATH)

if IN_COLAB and files:
    files.download(str(APP_MODEL_PATH))

## 15. Package Report Artifacts

Creates a zip file with the model charts/images for reports or presentations.

In [ ]:
artifact_patterns = ["*.png", "*.jpg", "*.jpeg"]
artifact_paths = []
for pattern in artifact_patterns:
    artifact_paths.extend(RUN_DIR.glob(pattern))

zip_path = PROJECT_ROOT / "report_images.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact in artifact_paths:
        archive.write(artifact, arcname=artifact.name)

print(f"Packaged {len(artifact_paths)} artifacts into {zip_path}")
if IN_COLAB and files:
    files.download(str(zip_path))

## 16. Next Steps

- Tune `EPOCHS`, `BATCH_SIZE`, and `BASE_MODEL` for accuracy/performance tradeoffs.
- Re-run test evaluation only after tuning decisions are final.
- Copy the exported `best.pt` into the SignGuard LK app project root before running `python app.py`.